# Fitting ion parameters of ePCP-SAFT

Simultaneous fit of the parameters of all ions to electrolyte data with exact derivatives (see `README.md`).

```
export EPCPSAFT_ELECTROLYTE_DATA=<data folder>      # e.g. ~/ePCP_dummy_data/data_epcpsaft
export EPCPSAFT_ION_START=<folder with ions.json, ions_binary.json, ions_options.json or truth_*.json>
export EPCPSAFT_ION_OUTPUT=<folder for the fitted parameters>   # default: results/ion_parameters (next to the notebook)
jupyter nbconvert --execute --to notebook --output-dir results adjust_ion_parameters.ipynb
```
Data and results are not part of the repository.

In [ ]:
import json, os
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import electrolyte_data as ed, electrolyte_fit as ef

DATA = os.environ["EPCPSAFT_ELECTROLYTE_DATA"]
START = os.environ["EPCPSAFT_ION_START"]
OUT = os.environ.get("EPCPSAFT_ION_OUTPUT", "results/ion_parameters")
VARIANT = "dh"                                       # ion-ion term: "dh" (Debye-Hückel) or "msa"
MSA_PACKING = "ions"                                 # MSA packing term over "ions" or "total" (all species)
N_STARTS, SCREENING_NFEV, N_POLISH = 4, 10, 2      # increase for real fits

## Data and target quantities

In [ ]:
df = ed.read(DATA)
points, skipped = ed.reduce(df)
print(f"{len(points)} points, {len(skipped)} skipped")
pd.DataFrame([(p.system, p.property, p.target, p.residual) for p in points],
             columns=["system", "property", "target", "residual"]).value_counts().rename("n").reset_index().head(20)

## Parameters and problem setup
Start values in feos format. One reference ion per group of connected ions (Cl⁻ where present) fixes the exact
symmetries; ε/k of Cl⁻ is set to 170 K (Held 2014) by the dispersion symmetry, which changes no prediction.
The optimizer works in the cross-energy variables (`Problem(cross_energies=True)`).

In [ ]:
def start_files(folder):
    for pure, binary, options in (("ions.json", "ions_binary.json", "ions_options.json"),
                                  ("truth_pure.json", "truth_binary.json", "options.json")):
        if os.path.exists(os.path.join(folder, pure)):
            return [os.path.join(folder, f) for f in (pure, binary, options)]
    raise FileNotFoundError(folder)

pure_file, binary_file, options_file = start_files(START)
pure, k_w = ef.read_records(pure_file, binary_file)
options = json.load(open(options_file))
ions = sorted({i for p in points for i in p.ions})
cp_blocks = sorted({ef.cp_block(p) for p in points if p.target == "Cp_phi"})
fixed = ef.gauge_fixed(points, pure)
par = ef.ParameterSet.new(ions, pure, k_w, ef.water_record(), (options["born_delta_cation"], options["born_delta_anion"]),
                          cp_blocks, fixed=fixed)
par = ef.reference_gauge(par, points, fixed)
print("reference ions:", sorted({i for i, _ in fixed}))
objective = ef.Objective(ef.Model(par, points, VARIANT, MSA_PACKING))
problem = ef.Problem(objective, par, cross_energies=True)
y_start = problem.to_y(par.x0[par.free])
print(f"{par.free.sum()} adjusted parameters; cost at start {problem.cost(y_start):.4g}")

## Optimization
Continuation (first low molalities near 298 K, then all data), multistart from the start values and a Sobol sample,
classification of the converged results, CMA-ES only if they are multimodal.

In [ ]:
easy = [p for p in points if sum(p.composition.values()) <= 1.0 and 273.0 <= p.T <= 323.2]
stage_1 = ef.Problem(ef.Objective(ef.Model(par, easy, VARIANT, MSA_PACKING)), par, cross_energies=True)
results = ef.continuation([stage_1, problem], n_starts=N_STARTS, starts=[y_start], screening_nfev=SCREENING_NFEV,
                          n_polish=N_POLISH, verbose=True)
kind, details = ef.classify(results, par)
print("classification:", kind, details)
best = results[0]
if kind == "multimodal":
    g = ef.cma_es(problem, best["y"], max_evals=2000)
    res = ef.fit(problem, g["y"])
    if res.cost < best["cost"]:
        best = {"y": res.x, "x": problem.x_free(res.x), "cost": res.cost}
costs, n_best = ef.waterfall(results)
print(f"best cost {best['cost']:.6g}; {n_best} of {len(costs)} starts within 0.1 %")
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.semilogy(np.arange(1, len(costs) + 1), costs, "o-")
ax.set_xlabel("start (sorted)"); ax.set_ylabel("final cost")
plt.tight_layout()

## Gauge (MacInnes convention), identifiability and errors

In [ ]:
x_best = par.full(best["x"])
if {"potassium ion", "chloride ion"} <= set(par.ions):
    x_best, c, residual = ef.macinnes_gauge(par, x_best, variant=VARIANT, msa_packing=MSA_PACKING)
    print(f"MacInnes shift of a11mu2 per unit charge: {c[0]:.4f}; max |ln gamma(K+) - ln gamma(Cl-)| {residual:.2e}")
d = ef.identifiability(objective, par, x_best)
print(f"condition number {d['condition']:.3g}")
for s, comb in d["weakest"]:
    print(f"  {s:.2e}", comb)

In [ ]:
m = ef.block_metrics(objective, par, x_best)
pd.DataFrame(m).T[["n", "failed"] + list(ef.METRICS)].round(5)

Deviations of the fitted quantities: relative deviations of the measured quantity ("relative", "linearized"),
deviations divided by the root mean square of the block ("scaled": V_φ, C_p,φ, Δ_dil H).

In [ ]:
r, calc = objective.relative_residuals(x_best)
props = sorted({p.property for p in points})
fig, axes = plt.subplots(len(props), 1, figsize=(7, 2.2 * len(props)), squeeze=False)
molality = np.array([sum(p.composition.values()) for p in points])
for ax, prop in zip(axes[:, 0], props):
    sel = np.array([p.property == prop for p in points])
    kind = points[np.flatnonzero(sel)[0]].residual
    ax.plot(molality[sel], 100 * r[sel], ".", ms=3)
    ax.axhline(0, color="k", lw=0.5)
    ax.set_ylabel("dev. / %" if kind != "scaled" else "dev. / rms / %"); ax.set_title(f"{prop} ({kind})", fontsize=9)
axes[-1, 0].set_xlabel("molality / (mol/kg)")
plt.tight_layout()

## Output (feos format)

In [ ]:
ef.write_parameters(par, x_best, OUT)
sorted(os.listdir(OUT))